Outline
This notebook demonstrates how to build a pipeline to predict results using Optuna hyperparameter lookup.

Preprocess and feature Engineering is base on previous notebook.

EDA link: [EDA of S6E3](https://www.kaggle.com/code/wesleyhuan/basic-eda-season-6-episode-3)

Key point: 
1. Data have a lot of non-numerical data in it. This requires to use OrdinalEncoder, OneHotEncoder and LabelEncoder depend on the data type.(Check out the link of **EDA of S6E3**)
2. This is a classification problem, but the submission we returned looks like this. This means we're trying to predict the probability of Churn. Therefore, you might need to use a different evaluation metric first, such as mean squared error (MSE) instead of the area under the curve (AUC).

Submission:

id,Churn

594194,0.1

594195,0.3

594196,0.2

etc.

Steps:

1. Load Data
2. Preprocess Data
3. Train model with Hyper parameter (OPTUNA)
4. Submission

# Load Data And Import Library

In [1]:
# import necessary library
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import OrdinalEncoder,LabelEncoder
from sklearn.preprocessing import OneHotEncoder,StandardScaler

from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

#for optuna pipline
import optuna
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.base import BaseEstimator, TransformerMixin
import torch

#deal with gender ratio prolem 2026/02/28
from sklearn.model_selection import StratifiedKFold

In [2]:
# config
class CFG:
    train_csv = '/kaggle/input/competitions/playground-series-s6e3/train.csv'
    test_csv = '/kaggle/input/competitions/playground-series-s6e3/test.csv'
    sample_submission_csv = '/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv'
    N_FOLDS = 5
    RANDOM_SEED = 42
    
#torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [3]:
train = pd.read_csv(CFG.train_csv)
test = pd.read_csv(CFG.test_csv)

In [4]:
train.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [5]:
test.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,594194,Female,0,Yes,No,72,Yes,Yes,Fiber optic,Yes,Yes,Yes,Yes,Yes,Yes,Two year,Yes,Electronic check,115.55,8061.50
1,594195,Female,0,Yes,No,71,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Bank transfer (automatic),19.80,1336.50
2,594196,Male,0,No,No,12,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Bank transfer (automatic),55.55,633.55
3,594197,Male,0,Yes,Yes,71,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,Two year,No,Credit card (automatic),84.10,6457.15
4,594198,Female,0,No,No,15,Yes,No,Fiber optic,Yes,No,No,No,Yes,Yes,Month-to-month,No,Electronic check,90.35,1233.65


# Preprocess Data

In [6]:
class Preprocessor:
    def __init__(self):
        self.medians = {}
        self.scaler = StandardScaler()
        self.one_hot_encoder = None
        
        self.binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
        self.replace_cols = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                             'TechSupport', 'StreamingTV', 'StreamingMovies']
        self.contract_mapping = {'Month-to-month': 0, 'One year': 1, 'Two year': 2}
        self.ohe_cols = ['InternetService', 'PaymentMethod']
        self.numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

    def create_interaction_features(self, df):
        df = df.copy()
        
        if 'TotalCharges' in df.columns:
            df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
        
        if 'MonthlyCharges' in df.columns and 'TotalCharges' in df.columns:
            df['Monthly_Fee_Ratio'] = df['MonthlyCharges'] / (df['TotalCharges'] + 1e-6)
        return df

    def fit(self, df, y=None):
        df_tmp = self.create_interaction_features(df)
        
        # learn median value
        curr_num_cols = df_tmp.select_dtypes(include=[np.number]).columns.tolist()
        for col in curr_num_cols:
            self.medians[col] = df_tmp[col].median()
            
        # fit OneHotEncoder
        self.one_hot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first')
        self.one_hot_encoder.fit(df_tmp[self.ohe_cols].astype(str))
        
        return self

    def transform(self, df):
        df = df.copy()
        df = self.create_interaction_features(df)
        
        if 'id' in df.columns: df = df.drop(columns=['id'])
        
        # Binary Encoding
        for col in self.binary_cols:
            if col in df.columns:
                df[col] = df[col].apply(lambda x: 1 if x in ['Yes', 'Male'] else 0)

        # No internet service and No phone service to No
        for col in self.replace_cols:
            if col in df.columns:
                df[col] = df[col].replace({'No internet service': 'No', 'No phone service': 'No'})
                df[col] = df[col].apply(lambda x: 1 if x == 'Yes' else 0)

        # Ordinal Encoding
        if 'Contract' in df.columns:
            df['Contract'] = df['Contract'].map(self.contract_mapping).fillna(0).astype(int)

        # fill median into miss value
        for col, val in self.medians.items():
            if col in df.columns:
                df[col] = df[col].fillna(val)

        # One-Hot Encoding (InternetService, PaymentMethod)
        encoded_array = self.one_hot_encoder.transform(df[self.ohe_cols].astype(str))
        encoded_cols = self.one_hot_encoder.get_feature_names_out(self.ohe_cols)
        encoded_df = pd.DataFrame(encoded_array, columns=encoded_cols, index=df.index, dtype=int)
        
        df = pd.concat([df, encoded_df], axis=1)
        df = df.drop(columns=self.ohe_cols)
        
        if 'Churn' in df.columns and isinstance(df['Churn'].iloc[0], str):
            df['Churn'] = df['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)

        return df

In [7]:
preprocessor = Preprocessor()

train_raw = train
X_train_raw = train.drop(columns=['id'])
y_train = train['Churn']

# learn the rule (median value...)
preprocessor.fit(X_train_raw)

# preprocess data
X_train_processed = preprocessor.transform(X_train_raw)

In [8]:
X_train_processed.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,...,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,Monthly_Fee_Ratio,InternetService_Fiber optic,InternetService_No,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,1,0,1,1,29,1,0,1,0,1,...,1,60.10,1653.85,0,0.036339,0,0,0,0,1
1,1,0,1,1,58,1,0,1,1,0,...,0,69.50,3778.20,0,0.018395,0,0,1,0,0
2,1,0,1,0,58,1,1,0,1,0,...,1,100.40,5841.35,0,0.017188,1,0,0,1,0
3,0,0,0,0,1,1,0,0,0,0,...,1,69.70,70.70,1,0.985856,1,0,0,1,0
4,0,0,0,0,1,1,0,0,0,0,...,1,70.45,70.45,1,1.000000,1,0,0,1,0


In [9]:
X_train_raw = train_raw.drop(columns=['Churn'])
y_train = X_train_processed['Churn']

# Train model with Hyper parameter (OPTUNA)

In [10]:
def XGB_objective(trial):
    model_params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 2000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }
    model_params["scale_pos_weight"] = trial.suggest_float("scale_pos_weight", 1.0, 5.0)
    
    pipeline = Pipeline(
        steps=[
            ("preprocess", Preprocessor()),
            ("model", XGBRegressor(
                objective="reg:logistic",
                random_state=CFG.RANDOM_SEED,
                n_jobs=-1,
                **model_params
            ))
        ]
    )

    scores = cross_val_score(
        pipeline,
        X_train_raw,
        y_train,
        cv=CFG.N_FOLDS,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1 
    )

    return -scores.mean()

In [11]:
study = optuna.create_study(direction="minimize")

# use Optuna to find best parameter
study.optimize(XGB_objective, n_trials=30)

[I 2026-03-24 14:20:56,532] A new study created in memory with name: no-name-eaf1baf1-33f0-422f-b4b8-2189881eb730
[I 2026-03-24 14:22:16,113] Trial 0 finished with value: 0.3819311082363129 and parameters: {'n_estimators': 789, 'max_depth': 3, 'learning_rate': 0.010571805507843484, 'subsample': 0.6142487256204707, 'colsample_bytree': 0.6954007810324208, 'min_child_weight': 3, 'reg_alpha': 0.37065890298075854, 'reg_lambda': 0.16551333579509017, 'scale_pos_weight': 4.991061903968921}. Best is trial 0 with value: 0.3819311082363129.
[I 2026-03-24 14:23:56,886] Trial 1 finished with value: 0.3651453971862793 and parameters: {'n_estimators': 663, 'max_depth': 8, 'learning_rate': 0.05838777420775207, 'subsample': 0.6294292157006185, 'colsample_bytree': 0.9062151112332641, 'min_child_weight': 2, 'reg_alpha': 0.03075111079167901, 'reg_lambda': 0.029138632550936034, 'scale_pos_weight': 4.659727116734095}. Best is trial 1 with value: 0.3651453971862793.
[I 2026-03-24 14:24:59,813] Trial 2 finish

In [12]:
print(f"Best AUC score: {study.best_value:.4f}")
print("Best parameters:", study.best_params)

Best AUC score: 0.3092
Best parameters: {'n_estimators': 1988, 'max_depth': 10, 'learning_rate': 0.013411385315033839, 'subsample': 0.8485239798057462, 'colsample_bytree': 0.5901134524372337, 'min_child_weight': 4, 'reg_alpha': 0.04276822772612425, 'reg_lambda': 1.0899002466658363e-08, 'scale_pos_weight': 1.0066898968965003}


# Submission

In [13]:
final_pipeline = Pipeline(
    steps=[
        ("preprocess", Preprocessor()),
        ("model", XGBRegressor(
            **study.best_params,
            objective="reg:logistic",
            random_state=CFG.RANDOM_SEED,
            n_jobs=-1
        ))
    ]
)

final_pipeline.fit(X_train_raw, y_train)
test_id = test["id"]
test = test.drop(columns=['id'])
y_pred = final_pipeline.predict(test)

In [14]:
submission = pd.read_csv(CFG.sample_submission_csv)

submission = pd.DataFrame({
    'id': test_id, 
    'Churn': y_pred   
})

submission.to_csv('submission.csv', index=False)

In [15]:
submission.head()

,id,Churn
0,594194,0.098712
1,594195,0.000254
2,594196,0.116532
3,594197,0.001704
4,594198,0.474446
